# Runtime Analysis - 1 Trial

# Install Library

In [1]:
!pip install mealpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 44.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.0 which is incompatibl

In [1]:
!pip install opfunu

# Import Library

In [2]:
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from copy import deepcopy

from mealpy import FloatVar
from mealpy import Multitask
from mealpy.utils.agent import Agent
from mealpy.optimizer import Optimizer
from mealpy.utils.problem import Problem
from mealpy.utils.termination import Termination
from mealpy.utils.validator import Validator

from typing import List, Tuple, Optional, Dict


# Connected to Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
path = "/content/drive/My Drive/Revision/"


# Define Necessary Function

## Benchmark Function

In [6]:
import sys
sys.path.append(path)

from benchmark_functions import classical_test_function
from benchmark_functions import cec20_test_function_with_bias
from benchmark_functions import cec22_test_function_with_bias

## Define Problems

In [7]:
# Classical Test Function
Fnames, Ffunctions, Fproblems, Flatexs = classical_test_function()

# Composite (CEC20) Test Function
C20names_50, C20functions_50, C20problems_50, C20latexs_50 = cec20_test_function_with_bias(dimension=50)
C20names_100, C20functions_100, C20problems_100, C20latexs_100 = cec20_test_function_with_bias(dimension=100)

# Composite (CEC22) Test Function
C22names_10, C22functions_10, C22problems_10, C22latexs_10 = cec22_test_function_with_bias(dimension=10)
C22names_20, C22functions_20, C22problems_20, C22latexs_20 = cec22_test_function_with_bias(dimension=20)


# Define Models

In [8]:
from optimizers import NHO
from optimizers import ACO, DE, GA
from optimizers import HGSO, GWO, HHO, SSO
from optimizers import ACSA, BPBO, CHO, SRA
from optimizers import L_SHADE, CMA_ES, IMODE

# Ek SOTA dosyasında kullanılan sınıf varsa dahil edilir.
try:
    from optimizers import LSHADEcnEpSin
    HAS_LSHADECNEPSIN = True
except Exception as e:
    HAS_LSHADECNEPSIN = False
    print("LSHADEcnEpSin import edilemedi, bu algoritma atlanacak:", e)


# Runtime Settings

In [9]:
trial = 1
n_workers = 4

SAVE_AS = "csv"
SAVE_CONVERGENCE = True
VERBOSE = True

runtime_save_root = path + "history/runtime_1trial_individual/"
os.makedirs(runtime_save_root, exist_ok=True)

SUITES = [
    {
        "suite": "classic",
        "dimension": 2,
        "folder": "classic",
        "problems": Fproblems,
        "epoch": 100,
        "pop_size": 30,
    },
    {
        "suite": "CEC22",
        "dimension": 10,
        "folder": "CEC22-10D",
        "problems": C22problems_10,
        "epoch": 200,
        "pop_size": 40,
    },
    {
        "suite": "CEC22",
        "dimension": 20,
        "folder": "CEC22-20D",
        "problems": C22problems_20,
        "epoch": 200,
        "pop_size": 50,
    },
    {
        "suite": "CEC20",
        "dimension": 50,
        "folder": "CEC20-50D",
        "problems": C20problems_50,
        "epoch": 400,
        "pop_size": 60,
    },
    {
        "suite": "CEC20",
        "dimension": 100,
        "folder": "CEC20-100D",
        "problems": C20problems_100,
        "epoch": 500,
        "pop_size": 80,
    },
]


# Helper Functions

In [10]:
def prepare_problems(problem_list):
    modified_problems = []

    for problem_dict in problem_list:
        new_problem_dict = problem_dict.copy()
        if "fit_func" in new_problem_dict:
            new_problem_dict["obj_func"] = new_problem_dict.pop("fit_func")
        modified_problems.append(new_problem_dict)

    return modified_problems


def build_algorithm_registry(epoch, pop_size):
    algorithms = {
        "NHO": NHO(epoch, pop_size),
        "ACO": ACO(epoch, pop_size),
        "DE": DE(epoch, pop_size),
        "GA": GA(epoch, pop_size),
        "GWO": GWO(epoch, pop_size),
        "HGSO": HGSO(epoch, pop_size),
        "HHO": HHO(epoch, pop_size),
        "SSO": SSO(epoch, pop_size),
        "ACSA": ACSA(epoch, pop_size),
        "BPBO": BPBO(epoch, pop_size),
        "CHO": CHO(epoch, pop_size),
        "SRA": SRA(epoch, pop_size),
        "L_SHADE": L_SHADE(epoch, pop_size),
        "CMA_ES": CMA_ES(epoch, pop_size),
        "IMODE": IMODE(epoch, pop_size),
    }

    if HAS_LSHADECNEPSIN:
        algorithms["LSHADEcnEpSin"] = LSHADEcnEpSin(epoch, pop_size)

    return algorithms


def run_single_algorithm_runtime(algo_name, algo, problems, save_path):
    os.makedirs(save_path, exist_ok=True)

    start_time = time.perf_counter()

    multitask = Multitask(
        algorithms=[algo],
        problems=problems,
        n_workers=n_workers
    )

    multitask.execute(
        n_trials=trial,
        save_path=save_path,
        save_as=SAVE_AS,
        save_convergence=SAVE_CONVERGENCE,
        verbose=VERBOSE
    )

    runtime = time.perf_counter() - start_time

    return runtime


# Run Runtime Analysis

In [13]:
runtime_results = []

for suite_cfg in SUITES:
    suite = suite_cfg["suite"]
    dimension = suite_cfg["dimension"]
    folder = suite_cfg["folder"]
    epoch = suite_cfg["epoch"]
    pop_size = suite_cfg["pop_size"]

    print("=" * 80)
    print(f"Suite: {suite} | Dimension: {dimension} | epoch={epoch} | pop_size={pop_size}")
    print("=" * 80)

    problems = prepare_problems(suite_cfg["problems"])
    algorithms = build_algorithm_registry(epoch, pop_size)

    for algo_name, algo in algorithms.items():
        print("=" * 80)
        print(f"Running: {algo_name} | {suite}-{dimension}D | trial={trial}")
        print("-" * 80)

        algo_save_path = os.path.join(runtime_save_root, folder, algo_name)

        try:
            runtime = run_single_algorithm_runtime(
                algo_name=algo_name,
                algo=algo,
                problems=problems,
                save_path=algo_save_path
            )

            runtime_results.append({
                "algorithm": algo_name,
                "suite": suite,
                "dimension": dimension,
                "folder": folder,
                "epoch": epoch,
                "pop_size": pop_size,
                "trial": trial,
                "n_workers": n_workers,
                "n_problems": len(problems),
                "runtime_seconds": runtime,
                "runtime_minutes": runtime / 60,
                "status": "OK",
                "error": ""
            })

            print(f"Runtime ({algo_name}, {folder}): {runtime:.4f} seconds")

        except Exception as e:
            runtime_results.append({
                "algorithm": algo_name,
                "suite": suite,
                "dimension": dimension,
                "folder": folder,
                "epoch": epoch,
                "pop_size": pop_size,
                "trial": trial,
                "n_workers": n_workers,
                "n_problems": len(problems),
                "runtime_seconds": np.nan,
                "runtime_minutes": np.nan,
                "status": "ERROR",
                "error": str(e)
            })

            print(f"ERROR ({algo_name}, {folder}): {e}")

runtime_df = pd.DataFrame(runtime_results)
runtime_df


Suite: classic | Dimension: 2 | epoch=100 | pop_size=30
Running: NHO | classic-2D | trial=1
--------------------------------------------------------------------------------
Solving problem: Ackley2 using algorithm: NHO, on the: 1 trial
Solving problem: Brent using algorithm: NHO, on the: 1 trial
Solving problem: ChungReynolds using algorithm: NHO, on the: 1 trial
Solving problem: Cigar using algorithm: NHO, on the: 1 trial
Solving problem: Matyas using algorithm: NHO, on the: 1 trial
Solving problem: Leon using algorithm: NHO, on the: 1 trial
Solving problem: Michalewicz using algorithm: NHO, on the: 1 trial
Solving problem: CrossInTray using algorithm: NHO, on the: 1 trial
Solving problem: Hosaki using algorithm: NHO, on the: 1 trial
Solving problem: Langermann using algorithm: NHO, on the: 1 trial
Solving problem: Levy5 using algorithm: NHO, on the: 1 trial
Solving problem: Mishra5 using algorithm: NHO, on the: 1 trial
Solving problem: Alpine2 using algorithm: NHO, on the: 1 trial
So

/content/drive/My Drive/Revision/optimizers.py:914: RuntimeWarning: overflow encountered in exp
  gama = self.beta * np.exp(-((self.p_best[idx].target.fitness + self.epsilon) / (self.pop_group[idx][jdx].target.fitness + self.epsilon)))


Solving problem: Langermann using algorithm: HGSO, on the: 1 trial
Solving problem: Levy5 using algorithm: HGSO, on the: 1 trial
Solving problem: Mishra5 using algorithm: HGSO, on the: 1 trial
Solving problem: Alpine2 using algorithm: HGSO, on the: 1 trial
Solving problem: Hansen using algorithm: HGSO, on the: 1 trial
Solving problem: Himmelblau using algorithm: HGSO, on the: 1 trial
Solving problem: Chichinadze using algorithm: HGSO, on the: 1 trial
Runtime (HGSO, classic): 6.6397 seconds
Running: HHO | classic-2D | trial=1
--------------------------------------------------------------------------------
Solving problem: Ackley2 using algorithm: HHO, on the: 1 trial
Solving problem: Brent using algorithm: HHO, on the: 1 trial
Solving problem: ChungReynolds using algorithm: HHO, on the: 1 trial
Solving problem: Cigar using algorithm: HHO, on the: 1 trial
Solving problem: Matyas using algorithm: HHO, on the: 1 trial
Solving problem: Leon using algorithm: HHO, on the: 1 trial
Solving prob

,algorithm,suite,dimension,folder,epoch,pop_size,trial,n_workers,n_problems,runtime_seconds,runtime_minutes,status,error
0,NHO,classic,2,classic,100,30,1,4,16,16.181175,0.269686,OK,
1,ACO,classic,2,classic,100,30,1,4,16,11.538174,0.192303,OK,
2,DE,classic,2,classic,100,30,1,4,16,5.686353,0.094773,OK,
3,GA,classic,2,classic,100,30,1,4,16,10.831807,0.180530,OK,
4,GWO,classic,2,classic,100,30,1,4,16,5.389977,0.089833,OK,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SRA,CEC20,100,CEC20-100D,500,80,1,4,10,258.454628,4.307577,OK,
76,L_SHADE,CEC20,100,CEC20-100D,500,80,1,4,10,331.625183,5.527086,OK,
77,CMA_ES,CEC20,100,CEC20-100D,500,80,1,4,10,2557.227447,42.620457,OK,
78,IMODE,CEC20,100,CEC20-100D,500,80,1,4,10,249.578789,4.159646,OK,


# Save Runtime Results

In [14]:
runtime_csv_path = path + "history/runtime_all_algorithms_1trial_individual.csv"
runtime_xlsx_path = path + "history/runtime_all_algorithms_1trial_individual.xlsx"

runtime_df.to_csv(runtime_csv_path, index=False)
runtime_df.to_excel(runtime_xlsx_path, index=False)

runtime_summary = (
    runtime_df
    .groupby("algorithm", as_index=False)
    .agg(
        total_runtime_seconds=("runtime_seconds", "sum"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        total_runtime_minutes=("runtime_minutes", "sum"),
        completed_runs=("status", lambda x: (x == "OK").sum()),
        failed_runs=("status", lambda x: (x == "ERROR").sum())
    )
    .sort_values("total_runtime_seconds")
)

runtime_summary_csv_path = path + "history/runtime_summary_all_algorithms_1trial.csv"
runtime_summary_xlsx_path = path + "history/runtime_summary_all_algorithms_1trial.xlsx"

runtime_summary.to_csv(runtime_summary_csv_path, index=False)
runtime_summary.to_excel(runtime_summary_xlsx_path, index=False)

print("Runtime detail CSV:", runtime_csv_path)
print("Runtime detail Excel:", runtime_xlsx_path)
print("Runtime summary CSV:", runtime_summary_csv_path)
print("Runtime summary Excel:", runtime_summary_xlsx_path)

runtime_summary


Runtime detail CSV: /content/drive/My Drive/Revision/history/runtime_all_algorithms_1trial_individual.csv
Runtime detail Excel: /content/drive/My Drive/Revision/history/runtime_all_algorithms_1trial_individual.xlsx
Runtime summary CSV: /content/drive/My Drive/Revision/history/runtime_summary_all_algorithms_1trial.csv
Runtime summary Excel: /content/drive/My Drive/Revision/history/runtime_summary_all_algorithms_1trial.xlsx


,algorithm,total_runtime_seconds,mean_runtime_seconds,total_runtime_minutes,completed_runs,failed_runs
15,SSO,292.957452,58.591490,4.882624,5,0
2,BPBO,293.150764,58.630153,4.885846,5,0
1,ACSA,315.496739,63.099348,5.258279,5,0
7,GWO,352.040441,70.408088,5.867341,5,0
8,HGSO,359.181550,71.836310,5.986359,5,0
5,DE,364.243399,72.848680,6.070723,5,0
10,IMODE,442.003695,88.400739,7.366728,5,0
14,SRA,465.502564,93.100513,7.758376,5,0
6,GA,472.480308,94.496062,7.874672,5,0
9,HHO,549.906490,109.981298,9.165108,5,0


# Optional Pivot Table

In [15]:
runtime_pivot = runtime_df.pivot_table(
    index="algorithm",
    columns="folder",
    values="runtime_seconds",
    aggfunc="mean"
)

runtime_pivot_path = path + "history/runtime_pivot_all_algorithms_1trial.csv"
runtime_pivot.to_csv(runtime_pivot_path)

print("Runtime pivot CSV:", runtime_pivot_path)
runtime_pivot


Runtime pivot CSV: /content/drive/My Drive/Revision/history/runtime_pivot_all_algorithms_1trial.csv


folder,CEC20-100D,CEC20-50D,CEC22-10D,CEC22-20D,classic
algorithm,,,,,
ACO,1673.096283,448.762844,71.975621,129.876360,11.538174
ACSA,169.038622,57.027286,32.541630,51.733933,5.155268
BPBO,159.266860,51.348674,30.379098,48.022050,4.134083
CHO,331.292052,130.523782,62.195756,95.378270,16.715441
CMA_ES,2557.227447,564.555477,86.824233,144.372931,21.480154
DE,200.895227,66.492834,35.156866,56.012119,5.686353
GA,259.310058,91.863672,42.961798,67.512973,10.831807
GWO,193.667663,65.298981,34.603637,53.080183,5.389977
HGSO,193.482561,63.662054,37.707562,57.689654,6.639720
